# Naive BC on PointMaze Medium

In [1]:
import random
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import PointMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000
seed = 0
lookback = 1
hidden_dims = {'L'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, L hidden
train_env = PointMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, L hidden
eval_env = PointMazePCH(num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = PointMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
naive_Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')
    naive_Z_sets[Xi] = cond

naive_Z_sets['X1']

{'P0', 'P1', 'W0', 'W1', 'X0'}

## Expert Trajectories

In [7]:
with open('/home/et2842/causal/expert_traj_pointmed.pkl', 'rb') as f:
    records = pickle.load(f)

print(f'loaded {len(records)} trajectories')

loaded 577619 trajectories


In [8]:
dims = {
    'P': 2,
    # 'L': 2,
    'W': 2,
    'X': 2
}

## Training

In [9]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 15
num_blocks = 4
epochs = 100
dropout = 0.0

In [10]:
nbc_model, nbc_slots, nbc_Z_trim = train_single_policy_long_horizon(
    records,
    naive_Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions=train_env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(train_env.action_space.low, train_env.action_space.high)
)

nbc_policy = shared_policy_fn_long_horizon(nbc_model, nbc_slots, nbc_Z_trim, continuous=True, device=device)
nbc_policies = make_shared_policy_dict(nbc_policy)

[LongHorizon] Epoch 1: train loss = 0.015076, val loss = 0.000437.


[LongHorizon] Epoch 2: train loss = 0.000342, val loss = 0.000390.


[LongHorizon] Epoch 3: train loss = 0.000249, val loss = 0.000157.


[LongHorizon] Epoch 4: train loss = 0.000214, val loss = 0.000134.


[LongHorizon] Epoch 5: train loss = 0.000170, val loss = 0.000133.


...

[LongHorizon] Epoch 30: train loss = 0.000057, val loss = 0.000047.


[LongHorizon] Epoch 31: train loss = 0.000043, val loss = 0.000031.


[LongHorizon] Epoch 32: train loss = 0.000055, val loss = 0.000038.


[LongHorizon] Epoch 33: train loss = 0.000039, val loss = 0.000039.


[LongHorizon] Early stop at epoch 34; best val 0.000026.


## Evaluation

In [11]:
num_eval_eps = 100
nbc_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=nbc_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(nbc_returns)

Starting episode 1/100...


  Episode 1 ended at step 1000 (terminated: False, truncated: True).
Starting episode 2/100...
  Episode 2 ended at step 103 (terminated: True, truncated: False).
Starting episode 3/100...


  Episode 3 ended at step 1000 (terminated: False, truncated: True).
Starting episode 4/100...
  Episode 4 ended at step 105 (terminated: True, truncated: False).
Starting episode 5/100...
  Episode 5 ended at step 106 (terminated: True, truncated: False).
Starting episode 6/100...


  Episode 6 ended at step 1000 (terminated: False, truncated: True).
Starting episode 7/100...
  Episode 7 ended at step 102 (terminated: True, truncated: False).
Starting episode 8/100...


  Episode 8 ended at step 1000 (terminated: False, truncated: True).
Starting episode 9/100...
  Episode 9 ended at step 105 (terminated: True, truncated: False).
Starting episode 10/100...
  Episode 10 ended at step 102 (terminated: True, truncated: False).
Starting episode 11/100...


...

  Episode 92 ended at step 105 (terminated: True, truncated: False).
Starting episode 93/100...
  Episode 93 ended at step 106 (terminated: True, truncated: False).
Starting episode 94/100...


  Episode 94 ended at step 1000 (terminated: False, truncated: True).
Starting episode 95/100...
  Episode 95 ended at step 105 (terminated: True, truncated: False).
Starting episode 96/100...
  Episode 96 ended at step 106 (terminated: True, truncated: False).
Starting episode 97/100...


  Episode 97 ended at step 105 (terminated: True, truncated: False).
Starting episode 98/100...
  Episode 98 ended at step 103 (terminated: True, truncated: False).
Starting episode 99/100...
  Episode 99 ended at step 104 (terminated: True, truncated: False).
Starting episode 100/100...


  Episode 100 ended at step 106 (terminated: True, truncated: False).
Finished collecting imitator trajectories.


48096

In [12]:
nbc_episode_rewards = defaultdict(float)
for rec in nbc_returns:
    ep = rec['episode']
    nbc_episode_rewards[ep] += float(rec['reward'])

nbc_rewards = [nbc_episode_rewards[e] for e in range(num_eval_eps)]
sum(nbc_rewards) / num_eval_eps

-359.2967325014156

## Save Model

In [13]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'nbc_pointmed.pt')

checkpoint = {
    "state_dict": nbc_model.state_dict(),
    "slots": nbc_slots,
    "Z_trim": nbc_Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": train_env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": dropout,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": eval_env.action_space.low,
    "action_bounds_high": eval_env.action_space.high,
    "input_dim": int(nbc_model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

Saved to: /home/et2842/causal/causalrl/models/nbc_pointmed.pt
